# 00 - Setup and environment check

**Purpose:** confirm the catalog, schema and volume, confirm both CSV files are visible,
and record the environment facts the README needs.

**What this notebook does NOT do:** it does not read, clean or transform any data.

**Rules**
- No credentials, tokens or secrets in this notebook.
- Catalogs are never created here. Schema/volume are only created if you set
  `create_missing_objects = yes` AND you have permission.
- Paste the printed output / screenshots into `evidence/01_workspace/`.

## 1. Parameters
Change the widget values at the top of the notebook if your names differ.

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "ecom_poc", "Schema")
dbutils.widgets.text("volume", "raw_files", "Volume")
dbutils.widgets.dropdown("create_missing_objects", "no", ["no", "yes"], "Create missing schema/volume?")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
VOLUME = dbutils.widgets.get("volume").strip()
CREATE_MISSING = dbutils.widgets.get("create_missing_objects") == "yes"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
EXPECTED_FILES = ["orders.csv", "transactions.csv"]

print(f"Catalog     : {CATALOG}")
print(f"Schema      : {SCHEMA}")
print(f"Volume      : {VOLUME}")
print(f"Volume path : {VOLUME_PATH}")
print(f"Expected    : {EXPECTED_FILES}")

## 2. Record the environment
Some values (region, trial dates) cannot always be read from code, especially on serverless.
Anything that prints `not available` must be filled in **manually** from the workspace UI.
Do not guess.

In [0]:
def safe_conf(key):
    """Read a Spark conf value; return None if the key is not available (common on serverless)."""
    try:
        return spark.conf.get(key)
    except Exception:
        return None


env = {}

# --- who / what are we running as ---
try:
    row = spark.sql(
        "SELECT current_user() AS user_name, current_catalog() AS cur_catalog, current_schema() AS cur_schema"
    ).collect()[0]
    env["current_user"] = row["user_name"]
    env["current_catalog"] = row["cur_catalog"]
    env["current_schema"] = row["cur_schema"]
except Exception as e:
    env["current_user"] = f"not available ({type(e).__name__})"

env["spark_version"] = spark.version

# current_version() is not available on every runtime, so it is wrapped
try:
    env["databricks_version"] = str(spark.sql("SELECT current_version()").collect()[0][0])
except Exception:
    env["databricks_version"] = "not available"

# --- workspace url -> cloud provider (inferred from the hostname pattern) ---
ws_url = safe_conf("spark.databricks.workspaceUrl")
env["workspace_url"] = ws_url or "not available"

if ws_url:
    if "azuredatabricks.net" in ws_url:
        env["cloud_provider_inferred"] = "Azure"
    elif "gcp.databricks.com" in ws_url:
        env["cloud_provider_inferred"] = "GCP"
    elif "cloud.databricks.com" in ws_url:
        env["cloud_provider_inferred"] = "AWS"
    else:
        env["cloud_provider_inferred"] = "unknown - check the workspace UI"
else:
    env["cloud_provider_inferred"] = "not available - check the workspace UI"

env["region"] = (
    safe_conf("spark.databricks.clusterUsageTags.region")
    or "not available - check the workspace UI / account console"
)

for k, v in env.items():
    print(f"{k:26s}: {v}")

## 3. Confirm the catalog exists

In [0]:
checks = {}  # name -> (passed: bool, detail: str); summarised at the end

catalogs = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
print("Catalogs visible to you:", catalogs)

checks["catalog_exists"] = (
    CATALOG in catalogs,
    f"catalog '{CATALOG}' {'found' if CATALOG in catalogs else 'NOT found'}",
)
print(checks["catalog_exists"][1])

if CATALOG not in catalogs:
    print("\nFix: change the 'catalog' widget to one of the catalogs listed above. "
          "Do not continue until this passes.")

## 4. Confirm (or create) the schema and volume
Creation only happens if `create_missing_objects = yes`. Permission errors are printed, not hidden.

In [0]:
def list_schemas(catalog):
    return [r[0] for r in spark.sql(f"SHOW SCHEMAS IN `{catalog}`").collect()]


def list_volumes(catalog, schema):
    """Use information_schema because SHOW VOLUMES column names differ between runtimes."""
    rows = spark.sql(f"""
        SELECT volume_name
        FROM `{catalog}`.information_schema.volumes
        WHERE volume_schema = '{schema}'
    """).collect()
    return [r["volume_name"] for r in rows]


if checks["catalog_exists"][0]:
    # ---- schema ----
    schemas = list_schemas(CATALOG)
    if SCHEMA not in schemas and CREATE_MISSING:
        try:
            spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
            print(f"Created schema {CATALOG}.{SCHEMA}")
            schemas = list_schemas(CATALOG)
        except Exception as e:
            print(f"Could not create schema - document this restriction:\n  {e}")

    checks["schema_exists"] = (SCHEMA in schemas, f"schema '{CATALOG}.{SCHEMA}' {'found' if SCHEMA in schemas else 'NOT found'}")
    print(checks["schema_exists"][1])

    # ---- volume ----
    if checks["schema_exists"][0]:
        volumes = list_volumes(CATALOG, SCHEMA)
        if VOLUME not in volumes and CREATE_MISSING:
            try:
                spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`")
                print(f"Created volume {CATALOG}.{SCHEMA}.{VOLUME}")
                volumes = list_volumes(CATALOG, SCHEMA)
            except Exception as e:
                print(f"Could not create volume - document this restriction:\n  {e}")

        checks["volume_exists"] = (VOLUME in volumes, f"volume '{VOLUME}' {'found' if VOLUME in volumes else 'NOT found'}")
        print(checks["volume_exists"][1])
        print("Volumes in schema:", volumes)

        # Screenshot-friendly evidence
        display(spark.sql(f"SHOW VOLUMES IN `{CATALOG}`.`{SCHEMA}`"))
else:
    print("Skipped: catalog check failed.")

## 5. Confirm both source files at the exact paths
Records name, full path, byte size and modified time. Take a screenshot of this output for
`evidence/01_workspace/`.

In [0]:
import datetime

file_records = []

if checks.get("volume_exists", (False, ""))[0]:
    listing = dbutils.fs.ls(VOLUME_PATH)
    by_name = {f.name: f for f in listing}

    print(f"Everything currently in {VOLUME_PATH}:")
    for f in listing:
        print(f"  {f.name:30s} {f.size:>15,d} bytes")

    for name in EXPECTED_FILES:
        full_path = f"{VOLUME_PATH}/{name}"
        if name in by_name:
            f = by_name[name]
            mtime_ms = getattr(f, "modificationTime", None)
            mtime = (
                datetime.datetime.fromtimestamp(mtime_ms / 1000, datetime.timezone.utc).isoformat()
                if mtime_ms else "not available"
            )
            file_records.append({
                "file": name,
                "path": full_path,
                "size_bytes": f.size,
                "size_mb": round(f.size / 1024 / 1024, 2),
                "modified_utc": mtime,
            })
            checks[f"file_{name}"] = (f.size > 0, f"{full_path} found, {f.size:,} bytes")
        else:
            checks[f"file_{name}"] = (False, f"{full_path} NOT found")
        print(checks[f"file_{name}"][1])

    if file_records:
        display(spark.createDataFrame(file_records))
else:
    print("Skipped: volume check failed.")

## 6. Confirm you can create tables in the schema
Bronze/Silver/Gold need CREATE TABLE rights. This creates and immediately drops a tiny test table.

In [0]:
TEST_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`_setup_permission_test`"

if checks.get("schema_exists", (False, ""))[0]:
    try:
        spark.sql(f"CREATE OR REPLACE TABLE {TEST_TABLE} AS SELECT 1 AS id")
        spark.sql(f"DROP TABLE IF EXISTS {TEST_TABLE}")
        checks["can_create_tables"] = (True, "create/drop table succeeded")
    except Exception as e:
        checks["can_create_tables"] = (False, f"create table failed: {e}")
    print(checks["can_create_tables"][1])
else:
    print("Skipped: schema check failed.")

## 7. Compute and warehouse inventory
Code cannot reliably list your compute, so record these **manually** from the UI
(Compute and SQL Warehouses pages). Fill the template in the next cell and keep it for the README.

In [0]:
readme_template = f"""
## Workspace environment (fill every blank from the UI - do not guess)

| Item | Value |
|---|---|
| Cloud provider | {env.get('cloud_provider_inferred')} |
| Region | {env.get('region')} |
| Workspace URL | (do not publish if sensitive) |
| Trial start date | ______ |
| Trial expiry date | ______ |
| Remaining trial credits (at start) | ______ |
| Notebook compute type (serverless / cluster) | ______ |
| SQL warehouse name and size | ______ |
| Runtime / Spark version | {env.get('databricks_version')} / {env.get('spark_version')} |
| Run by | {env.get('current_user')} |
| Catalog / schema / volume | {CATALOG} / {SCHEMA} / {VOLUME} |
| Observed restrictions (Genie, Jobs, connectors, grants...) | ______ |
"""
print(readme_template)

## 8. Readiness summary
Every line must say PASS before moving to `01_source_profile.py`.

In [0]:
print("=" * 70)
print("SETUP READINESS")
print("=" * 70)
for name, (ok, detail) in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {name:24s} {detail}")

all_ok = all(ok for ok, _ in checks.values()) and len(checks) > 0
print("=" * 70)
print("OVERALL:", "READY - continue to 01_source_profile" if all_ok else "NOT READY - fix the FAIL items above")

# Stop here (and fail a Lakeflow Job task) if anything is wrong
assert all_ok, "Setup checks failed - see the summary above."